In [ ]:
#!/usr/bin/env python3
"""
Simple test script for Corti transcription API.
Demonstrates uploading a single audio file and getting the transcript.
"""

from corti_transcript_client import CortiTranscriptClient
from dotenv import load_dotenv
import json
import os

load_dotenv()

print("=" * 70)
print("🎙️  CORTI SINGLE-FILE TRANSCRIPTION TEST")
print("=" * 70)

# Initialize client
client = CortiTranscriptClient()

# Step 1: Authenticate
print("\n1️⃣  Authenticating...")
try:
    token = client.authenticate()
    print(f"   ✅ Authenticated successfully")
except Exception as e:
    print(f"   ❌ Authentication failed: {e}")
    exit(1)

# Step 2: Create interaction
print("\n2️⃣  Creating interaction...")
try:
    interaction = client.create_interaction()
    interaction_id = interaction.get('interactionId') or interaction.get('id')
    print(f"   ✅ Interaction ID: {interaction_id}")
except Exception as e:
    print(f"   ❌ Failed to create interaction: {e}")
    exit(1)

# Step 3: Upload audio file
print("\n3️⃣  Uploading audio file...")
audio_path = os.path.join("Sample", "Patient Consultation with MI.mp3")

if not os.path.exists(audio_path):
    print(f"   ⚠️  File not found, trying alternative...")
    audio_path = os.path.join(os.path.dirname(__file__), "Sample", "TalkCPR- A real patient and doctor interaction filmed.mp3")

try:
    file_size = os.path.getsize(audio_path) / (1024 * 1024)  # MB
    print(f"   📁 File: {os.path.basename(audio_path)} ({file_size:.1f} MB)")

    with open(audio_path, 'rb') as f:
        upload = client.upload_recording(f, interaction_id)

    recording_id = upload.get('recordingId') or upload.get('id')
    print(f"   ✅ Recording ID: {recording_id}")
except Exception as e:
    print(f"   ❌ Upload failed: {e}")
    try:
        client.delete_interaction(interaction_id)
    except:
        pass
    exit(1)

# Step 4: Create transcript with options
print("\n4️⃣  Creating transcript...")
print("   📋 Options: Diarization enabled, Multi-channel, English")

try:
    transcript_response = client.create_transcript(
        interaction_id=interaction_id,
        recording_id=recording_id,
        primary_language="en",
        is_dictation=False,
        is_multichannel=True,
        diarize=True,
        participants=[{"channel": 1, "role": "doctor"}]
    )

    transcript_id = transcript_response.get('transcriptId') or transcript_response.get('id')
    print(f"   ✅ Transcript ID: {transcript_id}")
except Exception as e:
    print(f"   ❌ Failed to create transcript: {e}")
    if hasattr(e, 'response'):
        try:
            print(f"   Response: {e.response.json()}")
        except:
            print(f"   Response: {e.response.text}")
    # Clean up
    try:
        client.delete_recording(interaction_id, recording_id)
        client.delete_interaction(interaction_id)
    except:
        pass
    exit(1)

# Step 5: Wait for processing
print("\n5️⃣  Waiting for transcription to complete...")
print("   ⏳ This may take a few minutes...")

try:
    transcript = client.wait_for_transcript(
        interaction_id=interaction_id,
        transcript_id=transcript_id,
        max_wait_seconds=300,
        poll_interval=5
    )

    print(f"   ✅ Transcription completed!")

    # Display results
    print("\n" + "=" * 70)
    print("📊 TRANSCRIPTION RESULTS")
    print("=" * 70)

except TimeoutError as e:
    print(f"   ⏱️  {e}")
    print("   💡 Transcript is still processing. You can check status later.")
except Exception as e:
    print(f"   ❌ Error: {e}")


🎙️  CORTI SINGLE-FILE TRANSCRIPTION TEST

1️⃣  Authenticating...
   ✅ Authenticated successfully

2️⃣  Creating interaction...
   ✅ Interaction ID: f095749e-226f-40aa-b5d4-5e1c9c39f55f

3️⃣  Uploading audio file...
   📁 File: Patient Consultation with MI.mp3 (14.2 MB)
   ✅ Recording ID: 303c17ea-c9c7-4d17-b71f-3cfd3da3b1da

4️⃣  Creating transcript...
   📋 Options: Diarization enabled, Multi-channel, English
   ✅ Transcript ID: f4fa7144-b1f8-44fd-b664-00629065eee2

5️⃣  Waiting for transcription to complete...
   ⏳ This may take a few minutes...
   ✅ Transcription completed!

📊 TRANSCRIPTION RESULTS


In [34]:
transcript.keys()

dict_keys(['id', 'metadata', 'transcripts', 'usageInfo', 'recordingId', 'status'])

In [37]:
transcript['recordingId']

'303c17ea-c9c7-4d17-b71f-3cfd3da3b1da'

In [38]:
audio_path

'Sample/Patient Consultation with MI.mp3'

In [27]:
transcript['transcripts'][0]

{'channel': 0,
 'participant': 0,
 'speakerId': 0,
 'text': "We'll run through that scenario again, but this time we'll add in motivational interviewing stages.",
 'start': 560,
 'end': 4740}

In [ ]:
# Initialize as a list for better performance
transcript_parts = []

if 'transcripts' in transcript:
    for entry in transcript['transcripts']:
        # Check if the key exists and matches the channel
        if entry.get('channel') == 1 and 'text' in entry:
            transcript_parts.append(entry['text'])

# Join the list into a single string separated by a space (or newline)
transcript_text = " ".join(transcript_parts)

In [ ]:
# Show full transcript text
transcript_parts = []

if 'transcripts' in transcript:
    for entry in transcript['transcripts']:
        if entry.get('channel') == 1 and 'text' in entry:
            transcript_parts.append(entry['text'])

transcript_text = " ".join(transcript_parts)

print(f"\n📝 Full Transcript ({len(transcript_text)} characters):")
print("-" * 70)


📝 Full Transcript (5740 characters):
----------------------------------------------------------------------
----------------------------------------------------------------------


In [ ]:
folder_name = "transcripts_text"
file_name = "Sample" + "Patient Consultation with MI.mp3" + "full_transcript.txt"
file_path = os.path.join(folder_name, file_name)

# Create folder if it doesn't exist
if not os.path.exists(folder_name):
    os.makedirs(folder_name)
    print(f"📁 Created folder: {folder_name}")


In [44]:
# Write the transcript_text to the file
try:
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(transcript_text)
    print(f"✅ Transcript successfully saved to: {file_path}")
except Exception as e:
    print(f"❌ An error occurred: {e}")

✅ Transcript successfully saved to: transcripts_text/SamplePatient Consultation with MI.mp3full_transcript.txt


# Testing new Saving method 

In [ ]:
print(f"   📂 Saving full transcript text of {audio_path} to 'transcripts_text' folder...")

folder_name = "transcripts_text"
file_name = "Sample" + "Patient Consultation with MI.mp3" + "full_transcript.txt"
file_path = os.path.join(folder_name, file_name)

# Create folder if it doesn't exist
if not os.path.exists(folder_name):
    os.makedirs(folder_name)
    print(f"📁 Created folder: {folder_name}")


client.save_transcription_text(transcript=transcript)